# sub-16: recovering the bars sequence with the mglRetinotopy generator

**Clues** (supplementary material, sourcedata, mgl code v2.0):

1. Stimuli were made with `mglRetinotopy` (MGL v2.0): bars sweep in 8 directions (0:45:315°), 24 s per sweep, 10 sweeps.
2. Our two known sequences are both outputs of this generator:
   - **B** = default mode: the 8 directions in a fixed order, no blank.
   - **A** = `blanks` mode: initial half-cycle blank, then a **random permutation** of the 8 directions with one blank inserted. The order is random for each run unless `fixedRandom` is set.
3. Scan dates: sub-16 (jb, 2014-07-01) is the **first participant scanned**; sub-17 and sub-14 (B) come next (Aug 2014), then all others (A, from Oct 2014). Its last bars run (BarsLeftEye run-02) is A.

**Hypothesis**: the 3 suspect runs of sub-16 were made with the same generator, but each with a **new random permutation** (pilot session, before the sequence was fixed). That explains why they do not look like each other, or like A or B.

**Method**: instead of guessing a bar at each TR (previous notebook, 61 free apertures), the sequence is searched as a chain of **events** of the generator: a sweep of 12 TR in one of the 8 directions (frames copied from A and B), or a blank (5, 6 or 12 TR). Beam search on the full pRF prediction (canonical HRF, delay 0 to -2 TR). Far fewer free choices, so much more robust.

**Validation**:
- The sequence is estimated separately on V1+V3 and on V2. If both independent estimates give **the same chain of events**, the sequence is found.
- Cross-validated score (estimated on one set, tested on the other), as before.
- Controls with known sequence: sub-13 (A), sub-17 (B), sub-16 BarsLeftEye run-02 (A): the true chain must be recovered exactly.
- Null: rings runs of sub-16 (no bar shown): low score and unstable chains expected.

Read only. Recovered design matrices are saved in `Desktop/fig_od/sub16_recovered_mgl/`.

In [1]:
import os, glob, math
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
out_dir = '/Users/uriel/Desktop/fig_od/sub16_recovered_mgl'
os.makedirs(out_dir, exist_ok=True)
TR = 2.0
runs_to_do = [('sub-16', 'BarsLeftEye_run-01', 'suspect'), ('sub-16', 'BarsRightEye_run-01', 'suspect'),
              ('sub-16', 'BarsRightEye_run-02', 'suspect'), ('sub-16', 'BarsLeftEye_run-02', 'A'),
              ('sub-13', 'BarsLeftEye_run-01', 'A'), ('sub-13', 'BarsRightEye_run-02', 'A'),
              ('sub-17', 'BarsLeftEye_run-01', 'B'), ('sub-17', 'BarsRightEye_run-02', 'B'),
              ('sub-16', 'RingsLeftEye_run-01', 'null (rings)'), ('sub-16', 'RingsRightEye_run-01', 'null (rings)')]
beam_width = 200
n_vert_est = 600              # best vertices (pRF R2) of the estimation set used for the search
delays = [0, -1, -2]          # HRF delay tested (TR)
blank_lengths = [5, 6, 12]

/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_5162/3148998336.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## Known sequences and the 8 sweep templates

In [2]:
def build_vdm(mat_fn):
    """Same conversion as create_vdm_files.ipynb (50 x 50, transposed, padded)."""
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(ii, ii)]
    return out

A = np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedgesLeftEye_vdm.npy')[..., :120]
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
true_stim = {'A': A, 'B': B}

rr, cc = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc, -rr


def frame_label(f):
    m = f > 0
    if m.sum() < 3:
        return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a)
        p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]:
            best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    """stim (50,50,T) -> list of events (key, start, length); key = 'b' (blank) or '<ori><sign>'."""
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]
        u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            sign = '+' if lab[u - 1][1] > p else '-'
            ev.append((f'{o}{sign}', t, u - t))
        t = u
    return ev


templates = {}
for name, stim in [('A', A), ('B', B)]:
    ev = segment(stim)
    print(name, ' '.join(f'{k}({l})' for k, s, l in ev))
    for k, s, l in ev:
        if k != 'b' and l == 12 and k not in templates:
            templates[k] = stim[..., s:s + 12]
print('templates:', sorted(templates), '->', len(templates), 'directions')
true_events = {k: [(e[0], e[2]) for e in segment(v)] for k, v in true_stim.items()}

A b(6) 135+(12) 90+(12) b(5) 45-(12) 0+(12) 135-(12) 0-(12) 45+(12) 90-(12) 135-(12) 135-(1)
B 0+(12) 135-(12) 90-(12) 45-(12) 0-(12) 135+(12) 90+(12) 45+(12) 0+(12) 135-(12)
templates: ['0+', '0-', '135+', '135-', '45+', '45-', '90+', '90-'] -> 8 directions


## Data, pRFs and prediction

In [3]:
cache = {}
def load_subject(s):
    if s in cache: return cache[s]
    roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                          for h in ['L', 'R']])
    prf = {eye: np.hstack([np.vstack([a.data for a in nb.load(
        f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii').darrays])
        for h in ['L', 'R']]) for eye in ['LeftEye', 'RightEye']}
    cache[s] = (roi, prf)
    return cache[s]


def load_run(s, r):
    hs = []
    for h in ['L', 'R']:
        d = np.vstack([a.data for a in nb.load(
            f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
        hs.append(d if d.shape[0] == 120 else d.T)
    return np.hstack(hs)


pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
N_fft = 256
Hf = np.fft.rfft(H, n=N_fft)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=N_fft, axis=0) * Hf[:, None], n=N_fft, axis=0)[:d.shape[0]]


def shift(p, sh):
    q = np.roll(p, sh, 0)
    if sh > 0: q[:sh] = 0
    elif sh < 0: q[sh:] = 0
    return q


def zc(a, axis=0):
    a = a - a.mean(axis, keepdims=True)
    return a / (a.std(axis, keepdims=True) + 1e-9)


def drive(frames, x0, y0, s, n):
    """frames (k,50,50) -> (k, vertices)."""
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * s ** 2))
    return np.clip(frames.reshape(len(frames), -1) @ G, 0, None) ** n


def events_to_stim(events):
    st = np.zeros((50, 50, 120))
    t = 0
    for k, l in events:
        if k != 'b' and t < 120:
            n = min(12, 120 - t)
            st[..., t:t + n] = templates[k][..., :n]
        t += l
    return st


def test_score(stim, y, D_test):
    """Median correlation between data and prediction of the test vertices, best delay among -2..+2 TR."""
    x0, y0, s, n = D_test
    p = conv_hrf(drive(np.moveaxis(stim, -1, 0), x0, y0, s, n))
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(y)).mean(0))))
    return best

## Beam search over generator events

In [4]:
def window_score(pred, y, w):
    p, q = pred[:w], y[:w]
    p = p - p.mean(0); q = q - q.mean(0)
    den = np.sqrt((p ** 2).sum(0) * (q ** 2).sum(0))
    r = np.where(den > 0, (p * q).sum(0) / np.where(den > 0, den, 1), 0)
    return float(r.mean())


def search(y, D_est, delay):
    """y (120, n_vert); returns best events list and its full score."""
    x0, y0, s, n = D_est
    kern = {}
    for k, fr in templates.items():
        z = np.zeros((120, len(x0)), np.float32)
        z[:12] = drive(np.moveaxis(fr, -1, 0), x0, y0, s, n)
        kern[k] = shift(conv_hrf(z), delay).astype(np.float32)
    y = y.astype(np.float32)
    beam = [(np.zeros_like(y), l, [('b', l)] if l else [], 0.0) for l in range(13)]   # initial blank 0..12 TR
    done = []
    while beam:
        cand = []
        for pred, t, ev, sc in beam:
            last_blank = bool(ev) and ev[-1][0] == 'b'
            opts = [(k, 12) for k in kern] + ([] if last_blank else [('b', l) for l in blank_lengths])
            for k, l in opts:
                p2 = pred if k == 'b' else pred.copy()
                if k != 'b':
                    p2[t:] += kern[k][:120 - t]
                t2 = t + l
                ev2 = ev + [(k, l)]
                if t2 >= 120:
                    done.append((ev2, window_score(p2, y, 120), p2))
                else:
                    cand.append((p2, t2, ev2, window_score(p2, y, min(120, t2 + 3))))
        cand.sort(key=lambda z: -z[3])
        beam = cand[:beam_width]
        done.sort(key=lambda z: -z[1]); done = done[:beam_width]
    return done[0][0], done[0][1]


def ev_str(ev):
    return ' '.join(f'b{l}' if k == 'b' else k for k, l in ev)


def same_chain(e1, e2):
    """Same sweeps in the same order at the same times (+-1 TR)."""
    def sweeps(ev):
        t, out = 0, []
        for k, l in ev:
            if k != 'b' and t < 120: out.append((k, t))
            t += l
        return out
    a, b = sweeps(e1), sweeps(e2)
    return len(a) == len(b) and all(k1 == k2 and abs(t1 - t2) <= 1 for (k1, t1), (k2, t2) in zip(a, b))

In [5]:
rows, recovered = [], {}
for s, r, kind in runs_to_do:
    roi, prf = load_subject(s)
    eye = 'LeftEye' if 'LeftEye' in r else 'RightEye'
    D = prf[eye]
    y_all = load_run(s, r)
    ok = (D[0] > 0.2) & (D[1] < 6) & np.isfinite(y_all).all(0) & (y_all.std(0) > 0)
    sets = {'V1V3': np.where(ok & ((roi == 1) | (roi == 3)))[0], 'V2': np.where(ok & (roi == 2))[0]}
    par = lambda idx: (D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1))
    res = {'subject': s, 'run': r, 'true': kind}
    chains, cv = {}, []
    for est, tst in [('V1V3', 'V2'), ('V2', 'V1V3')]:
        idx = sets[est][np.argsort(-D[0][sets[est]])[:n_vert_est]]
        yz = zc(y_all[:, idx])
        best = max((search(yz, par(idx), dl) + (dl,) for dl in delays), key=lambda z: z[1])
        chains[est] = best
        cv.append(test_score(events_to_stim(best[0]), y_all[:, sets[tst]], par(sets[tst])))
        res[f'chain {est}'] = ev_str(best[0])
        res[f'delay {est}'] = best[2]
    res['same chain V1V3 / V2'] = same_chain(chains['V1V3'][0], chains['V2'][0])
    res['cv r recovered'] = np.mean(cv)
    for k, v in true_stim.items():
        res[f'cv r true {k}'] = np.mean([test_score(v, y_all[:, sets[t]], par(sets[t])) for t in ['V2', 'V1V3']])
        res[f'chain = {k}'] = same_chain(chains['V1V3'][0], true_events[k]) or same_chain(chains['V2'][0], true_events[k])
    sw = [k for k, l in chains['V1V3'][0] if k != 'b'][:8]
    res['distinct directions in first 8 sweeps'] = len(set(sw))
    stim_rec = events_to_stim(chains['V1V3'][0])
    recovered[(s, r)] = (chains, stim_rec)
    np.save(f'{out_dir}/{s}_{r}_recovered_vdm.npy', stim_rec)
    rows.append(res)
    print(s, r, 'done', flush=True)
tab = pd.DataFrame(rows).set_index(['subject', 'run'])
pd.set_option('display.max_colwidth', 200)
print('true A:', ev_str(true_events['A']))
print('true B:', ev_str(true_events['B']))
tab[['true', 'chain V1V3', 'chain V2']]

sub-16 BarsLeftEye_run-01 done
sub-16 BarsRightEye_run-01 done
sub-16 BarsRightEye_run-02 done
sub-16 BarsLeftEye_run-02 done
sub-13 BarsLeftEye_run-01 done
sub-13 BarsRightEye_run-02 done
sub-17 BarsLeftEye_run-01 done
sub-17 BarsRightEye_run-02 done
sub-16 RingsLeftEye_run-01 done
sub-16 RingsRightEye_run-01 done
true A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-
true B: 0+ 135- 90- 45- 0- 135+ 90+ 45+ 0+ 135-


true  \
subject run                                  
sub-16  BarsLeftEye_run-01         suspect   
        BarsRightEye_run-01        suspect   
        BarsRightEye_run-02        suspect   
        BarsLeftEye_run-02               A   
sub-13  BarsLeftEye_run-01               A   
        BarsRightEye_run-02              A   
sub-17  BarsLeftEye_run-01               B   
        BarsRightEye_run-02              B   
sub-16  RingsLeftEye_run-01   null (rings)   
        RingsRightEye_run-01  null (rings)   

                                                                 chain V1V3  \
subject run                                                                   
sub-16  BarsLeftEye_run-01    b4 90+ 0+ b6 45+ 135- 45- 135+ 90- 135- 0- 0+   
        BarsRightEye_run-01     b5 135- 0+ 135- 0- 135+ 90+ b5 0- 0+ 90- b5   
        BarsRightEye_run-02   b4 135- 0- 45+ 135+ 45- b6 135- 90+ 90- 0+ 0+   
        BarsLeftEye_run-02    b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 0-   
sub-13  BarsLeftEye_run-01    b5 135+ 90+ b5 0- 0+ 135- 0- 0+ 90- 135- 135+   
        BarsRightEye_run-02   b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 0-   
sub-17  BarsLeftEye_run-01            0+ 135- 90- 0- 0- 135+ 90+ 0+ 0+ 135-   
        BarsRightEye_run-02     b1 0+ 135- 90- 0- 0- 135+ 90+ 45+ 135- 135-   
sub-16  RingsLeftEye_run-01                b8 0- 0- 0- 0- 0- 0- 0- 0- 0- 0-   
        RingsRightEye_run-01     b2 90+ 90- 90+ 90+ 90+ 90+ 90+ 90+ 90+ 90+   

                                                                     chain V2  
subject run                                                                    
sub-16  BarsLeftEye_run-01      b4 90+ 0+ b6 45+ 135- 45- 135+ 90- 135- 0- 0+  
        BarsRightEye_run-01        b7 135- 0+ 135- 45- 135+ 90+ b5 0- 90+ 90-  
        BarsRightEye_run-02    b5 135- 0- 45+ 135+ 45- b5 135- 90+ 90- 0+ 90+  
        BarsLeftEye_run-02      b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- b5  
sub-13  BarsLeftEye_run-01     b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 45-  
        BarsRightEye_run-02     b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 0-  
sub-17  BarsLeftEye_run-01             0+ 135- 90- 45- 0- 135+ 90+ 0+ 0+ 135-  
        BarsRightEye_run-02           b1 0+ 135- 90- 0- 0- 0- 90+ 45+ 0+ 135-  
sub-16  RingsLeftEye_run-01                b8 0- 0- 0- 0- 90- 0- 0- 0- 45- b5  
        RingsRightEye_run-01  135+ b12 135+ 135+ 135+ 135+ 135+ 90+ 135+ 135+

## Decision

In [6]:
cols = ['true', 'same chain V1V3 / V2', 'chain = A', 'chain = B', 'distinct directions in first 8 sweeps',
        'cv r recovered', 'cv r true A', 'cv r true B', 'delay V1V3', 'delay V2']
print(tab[cols].round(3).to_string())
print("How to read:")
print("- Controls (A / B): the chain must equal the true one (chain = A or B True), and cv r recovered must be close to cv r true.")
print("- Null (rings): low cv r recovered and V1V3 / V2 chains that disagree.")
print("- Suspect runs: recovered if V1V3 and V2 give the same chain (independent vertices), with a cv r at the level of the controls")
print("  (sub-16 BarsLeftEye run-02 with A gives the level reachable for this subject), ideally with 8 distinct directions")
print("  (signature of a random permutation of mglRetinotopy). Design matrices saved in", out_dir)

                                      true  same chain V1V3 / V2  chain = A  chain = B  distinct directions in first 8 sweeps  cv r recovered  cv r true A  cv r true B  delay V1V3  delay V2
subject run                                                                                                                                                                                  
sub-16  BarsLeftEye_run-01         suspect                  True      False      False                                      7           0.377        0.146        0.055           0         0
        BarsRightEye_run-01        suspect                 False      False      False                                      5           0.355        0.040        0.003           0        -2
        BarsRightEye_run-02        suspect                 False      False      False                                      7           0.358        0.042        0.021           0         0
        BarsLeftEye_run-02               A        